# 🧪 Aula 21 — Prática: Servindo Modelos Pesados

**Grupo de Estudos em MLOps — CEIA/UFG**

Nesta prática você vai **medir, acelerar e servir** um modelo, percorrendo a mesma progressão da aula:

| Parte | Tema | Precisa de GPU? |
|---|---|---|
| 1 | Baseline: latência e percentis (p50/p95/p99) | Não (GPU opcional) |
| 2 | Batching: o trade-off latência × throughput | Não (GPU opcional) |
| 3 | Aceleração: ONNX Runtime + quantização INT8 | Não |
| 4 | Serving real: BentoML com *adaptive batching* | Não |
| 5 | LLMs: vLLM e *continuous batching* | **Sim** |

> ⚙️ **No Colab**: ative a GPU em `Ambiente de execução > Alterar tipo de ambiente de execução > T4 GPU`. Sem GPU, as partes 1–4 funcionam normalmente e a parte 5 é pulada automaticamente.
>
> 🐳 **Prefere rodar fora do Colab?** Todo o código desta prática existe também como código-fonte documentado em `atividade/src/`, com `Dockerfile` e `docker-compose.yml` — veja o `README.md` da atividade.

In [ ]:
# Instala as dependências (o Colab já traz o PyTorch instalado).
# Em ambiente local, prefira: pip install -r requirements.txt
%pip install -q "transformers>=4.44" "optimum[onnxruntime]>=1.21" "onnxruntime>=1.19" "bentoml>=1.3" "httpx>=0.27" matplotlib

In [ ]:
import time

import numpy as np
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {DEVICE}")

if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU  : {props.name}")
    print(f"VRAM : {props.total_memory / 1024**3:.1f} GB")
    # Guarde esses números: VRAM é o PRIMEIRO filtro na escolha de hardware.
else:
    print("⚠️ Sem GPU: as partes 1–4 rodam normalmente; a parte 5 (vLLM) será pulada.")

## Parte 1 — Baseline: medindo latência do jeito certo

Antes de otimizar qualquer coisa: **meça**. Duas regras que valem para qualquer benchmark de inferência:

1. **Warmup**: as primeiras execuções pagam custos que não se repetem (compilação de kernels, caches frios, alocação de memória). Descarte-as.
2. **Percentis, não média**: SLOs de latência são definidos em p95/p99 — é o que os usuários "azarados" experimentam. Um serviço com média de 50 ms e p99 de 3 s é um serviço ruim, e a média nunca mostraria isso.

Usaremos um modelo **pesado o suficiente para os efeitos aparecerem, leve o suficiente para o Colab**: DistilBERT fine-tunado para análise de sentimento (~67M parâmetros). Em produção real, troque mentalmente por um YOLO grande, um Whisper, um modelo de embeddings — a mecânica é a mesma.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_ID = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model_cpu = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).eval()

n_params = sum(p.numel() for p in model_cpu.parameters())
print(f"Parâmetros: {n_params / 1e6:.1f}M")
print(f"Memória dos pesos em FP32: ~{n_params * 4 / 1024**2:.0f} MB  (regra: parâmetros × bytes/precisão)")


def benchmark(fn, n_iters=50, warmup=5):
    """Executa `fn` repetidamente e retorna as latências individuais em ms."""
    for _ in range(warmup):
        fn()
    latencies = []
    for _ in range(n_iters):
        t0 = time.perf_counter()
        fn()
        latencies.append((time.perf_counter() - t0) * 1000)
    return np.array(latencies)


def report(name, latencies):
    p50, p95, p99 = (np.percentile(latencies, p) for p in (50, 95, 99))
    print(f"{name:<30} p50={p50:8.2f} ms   p95={p95:8.2f} ms   p99={p99:8.2f} ms")
    return p50

In [ ]:
TEXT = "Serving heavy models is all about latency and throughput trade-offs."
enc_cpu = tokenizer(TEXT, return_tensors="pt")


@torch.no_grad()
def infer_cpu():
    model_cpu(**enc_cpu)


baselines = {}
baselines["cpu"] = report("PyTorch CPU (batch=1)", benchmark(infer_cpu))

if DEVICE == "cuda":
    model_gpu = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).eval().to("cuda")
    enc_gpu = {k: v.to("cuda") for k, v in enc_cpu.items()}

    @torch.no_grad()
    def infer_gpu():
        model_gpu(**enc_gpu)
        # ESSENCIAL: chamadas CUDA são assíncronas. Sem synchronize(),
        # você mede o tempo de ENFILEIRAR o kernel, não de executá-lo.
        torch.cuda.synchronize()

    baselines["gpu"] = report("PyTorch GPU (batch=1)", benchmark(infer_gpu))
    print(f"\nSpeedup GPU vs CPU (batch=1): {baselines['cpu'] / baselines['gpu']:.1f}x")
    print("👉 Modesto, não? Com batch=1 a GPU fica quase ociosa. Guarde esse número para a Parte 2.")

## Parte 2 — Batching: a alavanca central do serving

GPUs (e até CPUs com SIMD) são máquinas de **paralelismo massivo**: processar 32 amostras juntas custa pouco mais do que processar 1. O batching é a ferramenta número 1 para aumentar **throughput**...

...mas cada amostra dentro do batch **espera** as outras — a latência individual sobe. Todo sistema de serving vive nesse trade-off.

Vamos medir a curva: latência e throughput para vários tamanhos de batch.

In [ ]:
BATCH_SIZES = [1, 2, 4, 8, 16, 32, 64]

active_model = model_gpu if DEVICE == "cuda" else model_cpu
texts_pool = [TEXT] * max(BATCH_SIZES)

rows = []
for bs in BATCH_SIZES:
    enc = tokenizer(texts_pool[:bs], return_tensors="pt", padding=True)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    @torch.no_grad()
    def step():
        active_model(**enc)
        if DEVICE == "cuda":
            torch.cuda.synchronize()

    p50 = float(np.percentile(benchmark(step, n_iters=20, warmup=3), 50))
    throughput = bs / (p50 / 1000)  # amostras por segundo
    rows.append({"batch": bs, "lat_ms": p50, "tput": throughput})
    print(f"batch={bs:>3}   latência(p50)={p50:8.2f} ms   throughput={throughput:9.1f} amostras/s")

In [ ]:
import matplotlib.pyplot as plt

batches = [r["batch"] for r in rows]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(batches, [r["tput"] for r in rows], marker="o")
ax1.set_xscale("log", base=2)
ax1.set_xlabel("tamanho do batch")
ax1.set_ylabel("amostras/s")
ax1.set_title(f"Throughput × batch ({DEVICE.upper()})")
ax1.grid(True, alpha=0.3)

ax2.plot(batches, [r["lat_ms"] for r in rows], marker="o", color="tab:orange")
ax2.set_xscale("log", base=2)
ax2.set_xlabel("tamanho do batch")
ax2.set_ylabel("latência p50 do batch (ms)")
ax2.set_title("Latência × batch")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 🔎 O que observar nos gráficos

- **Na GPU**: o throughput cresce quase linearmente com o batch (a latência quase não sobe até certo ponto) — a GPU estava **ociosa** com batch=1. É por isso que servir modelo pesado com "uma requisição por vez" queima dinheiro.
- **Na CPU**: o throughput satura rápido — a CPU já estava ocupada com batch pequeno.
- Em algum ponto a curva de throughput **achata** e a latência dispara: o hardware saturou. Batches maiores que isso só pioram a latência.

**Pergunta**: em produção as requisições chegam **uma a uma**, de clientes diferentes. Quem monta o batch? 👉 O **servidor de inferência** (dynamic/adaptive batching) — é exatamente o que faremos na Parte 4.

## Parte 3 — Aceleração: ONNX Runtime + quantização INT8

Antes de escalar horizontalmente (mais máquinas), esprema o hardware que você tem:

- **Exportar para ONNX**: converte o modelo para um grafo estático, executado por um runtime otimizado (fusão de operadores, sem overhead do Python/PyTorch).
- **Quantização dinâmica INT8**: pesos em 1 byte em vez de 4 → modelo ~4x menor e mais rápido em CPU.

> ⚠️ Quantização pode degradar a qualidade do modelo. Em um projeto real, **sempre** valide as métricas (accuracy/F1/etc.) da versão quantizada no seu conjunto de avaliação antes de promovê-la.

Este é o mesmo código de `src/export_onnx.py` da atividade.

In [ ]:
from pathlib import Path

from onnxruntime.quantization import QuantType, quantize_dynamic
from optimum.onnxruntime import ORTModelForSequenceClassification

FP32_DIR = Path("models/onnx")
INT8_DIR = Path("models/onnx-int8")
FP32_DIR.mkdir(parents=True, exist_ok=True)
INT8_DIR.mkdir(parents=True, exist_ok=True)

# 1) Exporta PyTorch -> ONNX (FP32)
ort_model = ORTModelForSequenceClassification.from_pretrained(MODEL_ID, export=True)
ort_model.save_pretrained(FP32_DIR)
tokenizer.save_pretrained(FP32_DIR)
tokenizer.save_pretrained(INT8_DIR)  # o serviço da Parte 4 usa o dir INT8

# 2) Quantiza os pesos para INT8
quantize_dynamic(
    model_input=FP32_DIR / "model.onnx",
    model_output=INT8_DIR / "model.onnx",
    weight_type=QuantType.QInt8,
)
(INT8_DIR / "config.json").write_bytes((FP32_DIR / "config.json").read_bytes())

for label, path in [("ONNX FP32", FP32_DIR / "model.onnx"), ("ONNX INT8", INT8_DIR / "model.onnx")]:
    print(f"{label}: {path.stat().st_size / 1024**2:7.1f} MB")

In [ ]:
import onnxruntime as ort

enc_np = tokenizer(TEXT, return_tensors="np")


def make_onnx_fn(model_path):
    """Cria a função de inferência para um grafo ONNX em CPU."""
    session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
    input_names = {i.name for i in session.get_inputs()}
    feeds = {k: v.astype(np.int64) for k, v in enc_np.items() if k in input_names}
    return lambda: session.run(None, feeds)


print("Comparação em CPU (batch=1):\n")
p50_torch = report("PyTorch CPU", benchmark(infer_cpu))
p50_fp32 = report("ONNX Runtime FP32", benchmark(make_onnx_fn(FP32_DIR / "model.onnx")))
p50_int8 = report("ONNX Runtime INT8", benchmark(make_onnx_fn(INT8_DIR / "model.onnx")))

print(f"\nSpeedup ONNX FP32 vs PyTorch: {p50_torch / p50_fp32:.2f}x")
print(f"Speedup ONNX INT8 vs PyTorch: {p50_torch / p50_int8:.2f}x")

### 🔎 Interpretação

- ONNX Runtime acelera mesmo em FP32 (grafo otimizado, sem overhead do framework de treino).
- INT8 acelera mais e corta o modelo para ~1/4 do tamanho — em modelos maiores, isso é a diferença entre **precisar de uma A100** e **rodar numa L4** (ou numa CPU!).
- **Na GPU**, o análogo desta etapa é o **TensorRT** (kernels compilados por GPU, FP16/FP8) e, para LLMs, formatos INT4 como **AWQ/GPTQ**. A lógica é a mesma: menos bytes por parâmetro → menos memória e mais velocidade.

## Parte 4 — Serving de verdade: BentoML com adaptive batching

Até aqui, o batch era montado **por nós**, no cliente. Em produção, cada cliente manda **uma** requisição — e é o **servidor de inferência** que precisa fundi-las em batches.

Vamos subir um serviço BentoML que:

1. Carrega o modelo **ONNX INT8** da Parte 3 (runtime enxuto, sem PyTorch);
2. Ativa **adaptive batching** (`batchable=True`): o servidor segura requisições por até `max_latency_ms` e as processa juntas.

O arquivo abaixo é uma cópia fiel de `src/service.py` — o mesmo que roda no Docker.

In [ ]:
%%writefile service.py
"""Serviço de inferência: BentoML + ONNX Runtime INT8 + adaptive batching.

Cópia fiel de atividade/src/service.py — consulte-o para a versão
com documentação completa.
"""
import json
import os
from pathlib import Path

import bentoml
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer

MODEL_DIR = Path(os.environ.get("MODEL_DIR", "models/onnx-int8"))
MAX_SEQ_LEN = 128


def _softmax(logits):
    exp = np.exp(logits - logits.max(axis=-1, keepdims=True))
    return exp / exp.sum(axis=-1, keepdims=True)


@bentoml.service(resources={"cpu": "2"}, traffic={"timeout": 30})
class SentimentService:

    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
        self.session = ort.InferenceSession(
            str(MODEL_DIR / "model.onnx"), providers=["CPUExecutionProvider"]
        )
        self.input_names = {i.name for i in self.session.get_inputs()}
        config = json.loads((MODEL_DIR / "config.json").read_text())
        self.id2label = {int(k): v for k, v in config.get("id2label", {}).items()} or {
            0: "NEGATIVE", 1: "POSITIVE"
        }

    @bentoml.api(
        batchable=True,     # <- funde requisições concorrentes em um batch
        max_batch_size=32,  # teto do batch dinâmico
        max_latency_ms=20,  # espera máxima para formar o batch
    )
    def classify(self, texts: list[str]) -> list[dict]:
        encoded = self.tokenizer(
            texts, padding=True, truncation=True,
            max_length=MAX_SEQ_LEN, return_tensors="np",
        )
        feeds = {
            name: array.astype(np.int64)
            for name, array in encoded.items()
            if name in self.input_names
        }
        probs = _softmax(self.session.run(None, feeds)[0])
        preds = probs.argmax(axis=-1)
        return [
            {"label": self.id2label[int(p)], "score": round(float(probs[i, p]), 4)}
            for i, p in enumerate(preds)
        ]

In [ ]:
import os
import subprocess

import httpx

# Sobe o servidor BentoML em segundo plano.
server = subprocess.Popen(
    ["bentoml", "serve", "service:SentimentService", "--port", "3000"],
    env={**os.environ, "MODEL_DIR": "models/onnx-int8"},
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Espera a readiness probe responder — em produção, é assim que um
# orquestrador (Kubernetes/compose) decide quando mandar tráfego.
for _ in range(60):
    try:
        if httpx.get("http://localhost:3000/readyz", timeout=2).status_code == 200:
            print("✅ Servidor pronto!")
            break
    except httpx.HTTPError:
        pass
    time.sleep(1)
else:
    server.terminate()
    raise RuntimeError("Servidor não subiu. Re-execute sem stdout/stderr=DEVNULL para ver os logs.")

# Smoke test
resp = httpx.post("http://localhost:3000/classify", json={"texts": ["what a great class!"]})
print(resp.json())

In [ ]:
import asyncio

SAMPLES = [
    "This class about model serving is amazing!",
    "The latency of this API is terrible.",
    "Quantization made my model so much faster.",
    "I hate waiting for cold starts.",
    "Continuous batching is a brilliant idea.",
    "The GPU ran out of memory again...",
    "Deploying with containers keeps things reproducible.",
    "My p99 latency exploded under load.",
]


async def load_test(n_requests=200, concurrency=1, url="http://localhost:3000/classify"):
    """Cada requisição leva UM texto — como fariam clientes independentes.
    O ganho em concorrência alta vem do adaptive batching DO SERVIDOR."""
    sem = asyncio.Semaphore(concurrency)
    async with httpx.AsyncClient(timeout=30.0) as client:

        async def one(i):
            async with sem:
                t0 = time.perf_counter()
                r = await client.post(url, json={"texts": [SAMPLES[i % len(SAMPLES)]]})
                r.raise_for_status()
                return (time.perf_counter() - t0) * 1000

        t0 = time.perf_counter()
        lats = await asyncio.gather(*(one(i) for i in range(n_requests)))
        elapsed = time.perf_counter() - t0

    rps = n_requests / elapsed
    print(
        f"concorrência={concurrency:<3}  RPS={rps:7.1f}  "
        f"p50={np.percentile(lats, 50):6.1f} ms  p95={np.percentile(lats, 95):6.1f} ms"
    )
    return rps


# Baseline: um cliente de cada vez (nenhuma chance de formar batch).
# (Notebooks suportam await direto na célula.)
rps_sequencial = await load_test(concurrency=1)

In [ ]:
# Agora 32 clientes simultâneos: o servidor funde as requisições
# em batches de até 32 antes de chamar o modelo.
rps_concorrente = await load_test(concurrency=32)

print(f"\n📈 Ganho de throughput com adaptive batching: {rps_concorrente / rps_sequencial:.1f}x")

In [ ]:
# Encerra o servidor antes de seguir para a Parte 5.
server.terminate()
server.wait(timeout=10)
print("Servidor encerrado.")

### 🔎 Interpretação

- Com **concorrência 1**, cada requisição vira um batch de tamanho 1 — o pior caso da Parte 2.
- Com **concorrência 32**, o servidor espera até `max_latency_ms=20` e funde as requisições: o throughput sobe muito, e a latência individual sobe pouco (ela inclui a espera do batch — o trade-off em ação).
- Esse é o **dynamic/adaptive batching**, presente no BentoML e no Triton. Experimente variar `max_latency_ms` (5, 20, 100) no `service.py` e observar o efeito em p50 e RPS.

> No caminho **Docker** da atividade, este mesmo serviço roda com `docker compose up sentiment-api` e o teste com `python -m src.load_test --concurrency 32`.

## Parte 5 (GPU) — LLMs: vLLM e continuous batching

Modelos autoregressivos geram **um token por vez**, e cada sequência termina num momento diferente. O *dynamic batching* clássico não funciona bem aqui — a solução é o **continuous batching** (vLLM, SGLang): requisições entram e saem do batch **a cada iteração** de geração, mantendo a GPU sempre cheia.

Vamos demonstrar com o vLLM e um LLM pequeno (Qwen2.5-0.5B), comparando gerar prompts **um por um** vs **todos de uma vez**.

> ⚠️ **Só roda com GPU** (no Colab: runtime T4). A instalação do vLLM demora alguns minutos e, no Colab, pode pedir reinício do runtime — por isso esta parte fica no fim do notebook. Se o runtime reiniciar, re-execute apenas a célula de instalação e siga daqui.

In [ ]:
import torch

if torch.cuda.is_available():
    !pip install -q vllm
else:
    print("Sem GPU — pule para a conclusão.")

In [ ]:
if torch.cuda.is_available():
    from vllm import LLM, SamplingParams

    llm = LLM(
        model="Qwen/Qwen2.5-0.5B-Instruct",
        dtype="half",                 # T4 não suporta bfloat16
        max_model_len=2048,           # contexto menor -> KV cache cabe na VRAM
        gpu_memory_utilization=0.85,  # fração da VRAM reservada (pesos + KV cache)
    )
    sampling = SamplingParams(max_tokens=128, temperature=0.7)
    print("vLLM pronto. Repare no log acima: veja quanto da VRAM foi reservado para o KV cache!")

In [ ]:
if torch.cuda.is_available():
    import time

    prompts = [
        f"Explique em uma frase, como se fosse a pergunta {i} de uma prova, "
        f"um conceito importante de MLOps."
        for i in range(32)
    ]

    # --- Cenário A: sequencial (8 prompts, um por vez) -----------------
    t0 = time.perf_counter()
    n_tokens = 0
    for p in prompts[:8]:
        out = llm.generate([p], sampling, use_tqdm=False)
        n_tokens += len(out[0].outputs[0].token_ids)
    seq_s = time.perf_counter() - t0
    seq_tps = n_tokens / seq_s
    print(f"sequencial : 8 prompts em {seq_s:6.1f} s  ->  {seq_tps:8.1f} tokens/s")

    # --- Cenário B: 32 prompts de uma vez (continuous batching) --------
    t0 = time.perf_counter()
    outs = llm.generate(prompts, sampling, use_tqdm=False)
    n_tokens = sum(len(o.outputs[0].token_ids) for o in outs)
    bat_s = time.perf_counter() - t0
    bat_tps = n_tokens / bat_s
    print(f"batch (32) : 32 prompts em {bat_s:6.1f} s  ->  {bat_tps:8.1f} tokens/s")

    print(f"\n📈 Throughput agregado: {bat_tps / seq_tps:.1f}x maior com continuous batching")

### 🔎 Interpretação

- No cenário sequencial, a GPU gera **uma sequência por vez**: a fase de *decode* é **memory-bound** (o gargalo é ler os pesos da VRAM a cada token), então quase toda a capacidade de computação fica ociosa.
- Com 32 prompts simultâneos, o **continuous batching** amortiza a leitura dos pesos entre todas as sequências: o throughput agregado (tokens/s) se multiplica, com pouco impacto na velocidade individual.
- É o mesmo trade-off das Partes 2 e 4 — mas gerenciado **a cada token gerado**, porque cada sequência termina num momento diferente.

> No caminho **Docker** da atividade, o vLLM sobe como um servidor OpenAI-compatible (`docker compose --profile gpu up vllm`) e o script `src/load_test_llm.py` mede **TTFT** (tempo até o primeiro token) e **TPOT** (tempo por token) via streaming — as duas métricas que definem a experiência de um chat.

## 🏁 Conclusão e reflexão

Você percorreu, em miniatura, a progressão completa da aula:

| Etapa | O que fizemos | Em produção seria... |
|---|---|---|
| Baseline | Medimos p50/p95/p99 com warmup | Benchmark antes de qualquer otimização |
| Batching | Curva latência × throughput | Dimensionamento de `max_batch_size` |
| Aceleração | ONNX + INT8 (~4x menor) | TensorRT, FP8, AWQ/GPTQ |
| Serving | BentoML + adaptive batching | Triton/BentoML atrás de um load balancer |
| LLM | vLLM + continuous batching | vLLM/SGLang num cluster com Dynamo |

**Para discutir no encontro:**

1. No seu experimento, qual `max_latency_ms` você escolheria para um SLO de p95 < 100 ms? E se o SLO fosse p95 < 30 ms?
2. A versão INT8 ficou mais rápida — mas como você provaria que ela **não piorou** a qualidade do modelo antes de colocá-la em produção?
3. Por que o ganho do batching foi muito maior na GPU do que na CPU?
4. O continuous batching multiplicou os tokens/s **agregados**. O que aconteceu com a latência de **cada** requisição individual? Quando isso seria inaceitável?
5. Este notebook rodou tudo em uma máquina. O que muda quando o modelo **não cabe** em uma GPU? (Dica: reveja tensor/pipeline parallelism e *disaggregated serving* no material do monitor.)

📂 **Não deixe de explorar a versão Docker** (`atividade/README.md`): é lá que a prática vira um deploy de verdade — imagem multi-stage, healthcheck, compose com profile de GPU.